In [ ]:
import os
from glob import glob
import os.path as op

import warnings
from IPython.display import HTML

# import math
# from math import ceil
# import matplotlib as mpl
# import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
import numpy.typing as npt
import numpy as np
import nibabel as nb
from numpy import linalg, polyfit

from nilearn.image import check_niimg, new_img_like
import pandas as pd
from scipy.stats import zscore, linregress
from scipy.optimize import curve_fit
from sklearn.decomposition import PCA, FastICA

# from tedana.stats import get_coeffs

from tedana import io, stats, utils
from tedana.utils import get_spectrum

# import tensorly as tl
# import tlviz
# from tensorly.decomposition import parafac, parafac2, CP
# import tensorly.contrib.sparse as stl
# from tensorly.contrib.sparse.decomposition import parafac as sparse_parafac
# from tensorly.cp_tensor import cp_to_tensor
# from tensorly.random import random_cp

import sys
sys.path.append("/Users/smithmam/Documents/code/nimh-sfim/simlib/utils_tt.py")

from utils_tt import file_to_2d, mask_data, plot_time_and_echoes_brain, plot_echo_profiles,compare_rsqared, pca_dim_reduced, get_new_slopes, initialize_cp_with_ica, parafac_iter, ica_to_parafac_iter 

sys.path.append("/Users/smithmam/Documents/code/nimh-sfim/simlib/utils_tt.cpython-312.py")

import generate_sims as gen_sims

#Run tedana on a dataset with X components (X=30?)

# Generate OC data! Done for subj 1----
# cd /Users/smithmam/data/5_echo_checkerboard


tedana -d p05.SBJ01_S09_Task11_e1.in.nii.gz \
           p05.SBJ01_S09_Task11_e2.in.nii.gz \
           p05.SBJ01_S09_Task11_e3.in.nii.gz \
           p05.SBJ01_S09_Task11_e4.in.nii.gz \
           p05.SBJ01_S09_Task11_e5.in.nii.gz \
         --mask mask.nii.gz \
         -e 15.4 29.7 44.0 58.3 72.6 \
         --tree minimal \
         --tedpca 30 \
         --out-dir tedana_out_for_sim_s09_t11 --overwrite


In [ ]:
#Load the ICA mixing matrix (desc-ICA_mixing.tsv), the ICA weight maps (desc-ICA_components.nii.gz) and the component metrics table (desc-tedana_metrics.tsv)

drive_loc = "/Volumes/SFIM_MLT/handwerkerd/checkerboard2share/SBJ01"
#drive_loc = "/Users/smithmam/data/5_echo_checkerboard"

mixing_file = op.join(drive_loc, 'tedana_out_for_sim/desc-ICA_mixing.tsv')
comps_file = op.join(drive_loc, 'tedana_out_for_sim/desc-ICA_components.nii.gz')
metrics_file = op.join(drive_loc, 'tedana_out_for_sim/desc-tedana_metrics.tsv')
s0_file = op.join(drive_loc, 'tedana_out_for_sim/S0map.nii')
t2s_file = op.join(drive_loc, 'tedana_out_for_sim/T2starmap.nii')


# mixing_file = op.join(drive_loc, 'tedana_out_for_sim_s09_t11/desc-ICA_mixing.tsv')
# comps_file = op.join(drive_loc, 'tedana_out_for_sim_s09_t11/desc-ICA_components.nii.gz')
# metrics_file = op.join(drive_loc, 'tedana_out_for_sim_s09_t11/desc-tedana_metrics.tsv')

mixing = np.loadtxt(mixing_file, delimiter='\t', skiprows=1)
comps_input = nb.load(comps_file)
comps_img = comps_input.get_fdata()
comps_header = comps_input.header
print(comps_header)
x, y, z, a = comps_img.shape
comps = comps_img.reshape(x*y*z, a) # .astype(int)
non_zero_comps = np.sum(comps!=0, axis=1)>0
# reduce comps to only those with non-zero values
comps = comps[non_zero_comps,:]
comps = comps-np.mean(comps)
# Scale comps so that the maximum absolute value of the sum across components is 0.1
# (this is to ensure that the simulated signal changes are on the order of 10% or less, which is more realistic for fMRI data)
comps = comps/(0.1*np.max(np.abs(np.sum(comps, axis=1))))


s0_input = nb.load(s0_file)
s0_img = s0_input.get_fdata().reshape(x*y*z)
s0_img = s0_img[non_zero_comps]
t2s_input = nb.load(t2s_file)
r2s_img = 0.001/t2s_input.get_fdata().reshape(x*y*z)
r2s_img[r2s_img==np.inf] = 0
r2s_img = r2s_img[non_zero_comps]

s0_mean = np.mean(s0_img[s0_img>0])
r2s_mean = np.mean(r2s_img[r2s_img>0])
print(f"{s0_mean=}, {r2s_mean=}")


# np.loadtxt(comps_file, delimiter='\t', skiprows=1)

# df = pd.read_csv('tester.csv')
# df_to_array = np.array(df)

metrics_df = pd.read_csv(metrics_file, delimiter = "\t")
# metrics = np.array(metrics_df)
print(mixing.shape)
print(f"{comps.shape=}, {s0_img.shape=}, {r2s_img.shape=}")
print(f"{np.std(comps)=}, {np.mean(comps)=}, {np.max(comps)=}, {np.min(comps)=}")
plt.plot(comps.mean(axis=1))
plt.show()
plt.plot(comps.std(axis=1))
plt.show()
# print(metrics_df)

In [ ]:
#For each component, proportion_s0_r2s=1 if the “classification” column is “rejected” and 0 of “accepted"
n_comps = 30

proportion_s0_r2s_vec = np.zeros(n_comps)
print(proportion_s0_r2s_vec)
for component in range(n_comps):
    if metrics_df["classification"][component] == "rejected":
        proportion_s0_r2s_vec[component] = 1 #0.98 ** component

print(proportion_s0_r2s_vec)

In [ ]:
max = np.max(mixing)
print(max)

inputted_s = mixing/max*0.1
# s0_mean = 500 
# r2s_mean = 1/40
te_baseline=30

proportion_s0_r2s = np.tile(proportion_s0_r2s_vec, (inputted_s.shape[0], 1))
print(proportion_s0_r2s)
print(proportion_s0_r2s.shape)
print(f"{inputted_s.shape=}")
print(f"{inputted_s.mean(axis=0)=}")
print(f"{inputted_s.std(axis=0)=}")
print(f"{inputted_s.max(axis=0)=}")
print(f"{inputted_s.min(axis=0)=}")

In [ ]:
n_vox = len(s0_img)
n_time = inputted_s.shape[0]
s0_mean_tiled = np.tile(s0_img,[160,30,1]).transpose([2,0,1])
r2s_mean_tiled = np.tile(r2s_img,[160,30,1]).transpose([2,0,1])
inputted_s_tiled = np.tile(inputted_s,[n_vox,1,1])
proportion_s0_r2s_tiled = np.tile(proportion_s0_r2s,[n_vox,1,1])
comps_tiled = np.tile(comps, [n_time,1,1]).transpose([1,0,2])
print(r2s_mean_tiled.shape)
print(inputted_s_tiled.shape)
print(proportion_s0_r2s_tiled.shape)
print(comps_tiled.shape)

delta_s0, delta_r2s = gen_sims.calc_delta_r2s_s0_given_s_pchange_proportion(
        inputted_s=inputted_s_tiled,
        s0_mean=s0_mean_tiled,
        r2s_mean=r2s_mean_tiled,
        proportion_s0_r2s=proportion_s0_r2s_tiled,
        te_baseline=te_baseline,
        prop_to_scale= "variance")

voxelwise_delta_s0 = np.mean(comps_tiled*delta_s0, axis=2)
voxelwise_delta_r2s = np.mean(comps_tiled*delta_r2s, axis=2)
print(f"{voxelwise_delta_s0.shape=} {voxelwise_delta_r2s.shape=}")
print(f"{np.mean(voxelwise_delta_s0)=} {np.std(voxelwise_delta_s0)=}, {np.max(voxelwise_delta_s0/s0_mean_tiled[:,:,0])=}, {np.min(voxelwise_delta_s0/s0_mean_tiled[:,:,0])=}")
print(f"{np.mean(voxelwise_delta_r2s)=} {np.std(voxelwise_delta_r2s)=}, {np.max(voxelwise_delta_r2s/r2s_mean_tiled[:,:,0])=}, {np.min(voxelwise_delta_r2s/r2s_mean_tiled[:,:,0])=}")


tes=[10,20,30,40,50]

for te in tes:

    print(te)
    new_data = gen_sims.monoexponential(tes=te, s0=s0_mean_tiled[:,:,0]+ voxelwise_delta_s0, r2star=r2s_mean_tiled[:,:,0]+ voxelwise_delta_r2s)
#    new_mean_s = gen_sims.monoexponential(tes=te, s0=s0_mean_tiled[:,:,0], r2star=r2s_mean_tiled[:,:,0])

#     new_mixing = gen_sims.monoexponential(tes=te, s0=s0_mean_tiled+ delta_s0, r2star=r2s_mean_tiled+ delta_r2s)
#     tmp_mixing_mean = np.mean(new_mixing,axis=1, keepdims=True)
#     new_mixing_pchange = (new_mixing-tmp_mixing_mean)/tmp_mixing_mean
    # new_data = (np.sum(comps_tiled*new_mixing_pchange, axis=2)+1)*new_mean_s
#     new_data = (np.sum(comps_tiled*new_mixing_pchange, axis=2)) + new_mean_s
    #new_data = np.sum(comps_tiled*new_mixing, axis=2)

    # mixing matrix with the mean decay removed from each so that it doesn't scale across components
#    new_mixing = gen_sims.monoexponential(tes=te, s0=s0_mean_tiled+ delta_s0, r2star=r2s_mean_tiled+ delta_r2s)-np.tile(new_mean_s,(n_comps,1,1)).transpose((1,2,0))
    # Dot product of component weight maps and new mixing matrix and add a single mean back in
#    new_data = (np.sum(comps_tiled*new_mixing, axis=2)) + new_mean_s


#    print(f"{np.mean(new_mixing)=} {np.std(new_mixing)=}")
    # print(f"{np.mean(new_mixing_pchange)=} {np.std(new_mixing_pchange)=}")
#    print(f"{np.mean(inputted_s)=} {np.std(inputted_s)=}")

    print(x, y, z)
    new_data_full_shape = np.zeros((x*y*z, 160))
    new_data_full_shape[non_zero_comps>0,:] = new_data
    array = np.reshape(new_data_full_shape, (x, y, z, 160))

    print(array.shape)
    new_image = nb.Nifti1Image(array, affine=comps_header.get_best_affine(), header = comps_header)

    nb.save(new_image, op.join(op.abspath("/Users/handwerkerd/Downloads"), f"test_vol_{te}"))
    # nb.save(new_image, op.join("/Users/smithmam/data/5_echo_checkerboard", f"data_echo_time_{te}_var_with_one"))

In [ ]:
#new_data[:,tidx] = np.dot(comps, np.squeeze(new_mixing[:,tidx,:]).T)
print(np.tile(new_mean_s,(n_comps,1,1)).transpose((1,2,0)).shape)
print(comps.shape)
plt.plot(comps.mean(axis=1))
plt.show()
print(np.squeeze(new_mixing[:,tidx,:]).T.shape)
print(new_data.shape)
print(np.sum(comps*np.squeeze(new_mixing[:,tidx,:]),axis=1).shape)

In [ ]:
new_mixing.shape
tmp_mixing_mean = np.mean(new_mixing,axis=1, keepdims=True)
new_mixing_pchange = (new_mixing-tmp_mixing_mean)/tmp_mixing_mean
print(f"{np.nanmean(new_mixing_pchange)=}, {np.nanstd(new_mixing_pchange)=}")

In [ ]:

# print(inputted_s.shape, proportion_s0_r2s.shape)

# delta_s0, delta_r2s = gen_sims.calc_delta_r2s_s0_given_s_pchange_proportion(
#         inputted_s=inputted_s,
#         s0_mean=s0_mean,
#         r2s_mean=r2s_mean,
#         proportion_s0_r2s=proportion_s0_r2s,
#         te_baseline=te_baseline,
#         prop_to_scale= "variance")

# # data_shape = x, y, z, 160

# tes=[10,20,30,40,50]

# for te in tes:
#     print(te)
#     new_mixing = gen_sims.monoexponential(tes=te, s0=s0_mean+ delta_s0, r2star=r2s_mean+ delta_r2s)
# #    mean = np.mean(new_mixing_unzscored, axis = 0, keepdims=True)
# #    new_mixing = (new_mixing_unzscored-mean)/mean
#     # print(np.sum(new_mixing), np.sum(comps))
#     # print(new_mixing.shape, comps.shape)
#     new_data = np.dot(comps, new_mixing.T)

#     print(np.std(new_mixing))
#     print(np.std(mixing))
#     # print(np.sum(
#     # new_data))
#     # print(new_data.shape)

#     print(x, y, z)
#     array = np.reshape(new_data, (x, y, z, 160))

#     print(array.shape)
#     new_image = nb.Nifti1Image(array, affine=comps_info.get_best_affine(), header = comps_info)

#     nb.save(new_image, op.join(op.abspath("/Users/handwerkerd/Downloads"), "test_vol"))
#     # nb.save(new_image, op.join("/Users/smithmam/data/5_echo_checkerboard", f"data_echo_time_{te}_var_with_one"))
    


In [ ]:
print(new_data.shape)
#plt.plot(np.std(new_data,axis=0)/np.mean(new_data,axis=0))
#plt.plot(new_data[20000:20010,:].T/np.mean(new_data[20000:20010,:],axis=1))
plt.scatter(new_data[20000,:], new_data[20010,:])

In [ ]:
# for n in range(30):
#     plt.plot(mixing[:,n])
#     plt.show()

#     plt.plot(new_mixing_unzscored[:,n])
#     plt.show()

#     plt.plot(new_mixing[:,n])
#     plt.show()

In [ ]:
plt.plot(delta_s0.T)
plt.show()
plt.plot(delta_r2s.T)
plt.show()

In [ ]:
# comps = comps_img.reshape(x*y*z, a).astype(int)
# validate = np.reshape(comps, (x, y, z, 30))

# new_image = nb.Nifti1Image(validate, np.eye(4), dtype = np.int16)

# nb.save(new_image, op.join("/Users/smithmam/data/5_echo_checkerboard", "validate"))
    

# # comps = comps_img.reshape(x*y*z, a).astype(int) == 5

# # #     nb.save(new_data, f'simulation_data_te_{te}.nii.gz')
    




tedana -d data_echo_time_10_var.nii \
          data_echo_time_20_var.nii \
          data_echo_time_30_var.nii \
          data_echo_time_40_var.nii \
          data_echo_time_50_var.nii \
         --mask mask.nii.gz \
         -e 10 20 30 40 50 \
         --tree minimal \
         --tedpca 30 \
         --out-dir tedana_out_for_sim_results_var_varied --overwrite